# Email agent with Tool
- Here we add a tool that takes company internal terminalogy / jargon and replaces with professional words  

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load variables from the local .env file
load_dotenv()

# Check if the Gemini API key is already loaded in the environment
gemini_api_key = os.getenv("GEMINI_API_KEY")

if not gemini_api_key:
    print("⚠️ GEMINI_API_KEY not found in environment variables.")
    # Prompt the user for the key securely without displaying the typed characters
    gemini_api_key = getpass("🔐 Please enter your Gemini API Key: ")
    
    # Optional: Manually set it into the environment context for tracking libraries
    os.environ["GEMINI_API_KEY"] = gemini_api_key

print("✅ Gemini API Key is ready for use.")

✅ Gemini API Key is ready for use.


In [3]:
from crewai import LLM

llm = LLM(
    model="gemini/gemini-3.6-flash", # this model may get obsolete in the future
    temperature=0.1
)

# Test model api key
response = llm.call("Give me a one line joke about geometry")

print(response)

Parallel lines have so much in common, it’s a shame they’ll never meet.


In [7]:
from crewai.tools import BaseTool

class ReplaceJargonsTool(BaseTool):
    name: str = "Jargon replacement tool"
    description : str = "Replaces jargon with more specific terms. "

    def _run(self, email: str) -> str:
        replacements = {
            "PRG": "Project Gemini",
            "BTW": "by the way",
            "ETA": "estimated time of arrival",  
            "TA": "technical architecture",
            "DB": "database",
            "WIP": "work in progress",
            "POC": "proof of concept",
            "ping": "reach out"
        }
        suggestions = []
        email_lower = email.lower()
        for jargon, replacement in replacements.items():
            if jargon.lower() in email_lower:
                suggestions.append(f"Consider replacing '{jargon}' with '{replacement}'")

        return "\n".join(suggestions) if suggestions else "No jargon or internal abbreviations detected."

jt = ReplaceJargonsTool()

original_email = """
looping in John. TA and PRG updates are in the deck. ETA for PRG integration is Fri.
Let's sync up tomorrow. BTW, ping me if any hurdles.
"""

jt.run(original_email)

"Consider replacing 'PRG' with 'Project Gemini'\nConsider replacing 'BTW' with 'by the way'\nConsider replacing 'ETA' with 'estimated time of arrival'\nConsider replacing 'TA' with 'technical architecture'\nConsider replacing 'ping' with 'reach out'"

In [6]:
from crewai import Agent, Task, Crew

email_assistant = Agent(
    role="Email Assistant Agent",
    goal="Improve emails and make them sound professional and clear",
    backstory="A highly experienced communication expert skilled in professional email writing",
    verbose=True,
    tools=[jt],
    llm=llm
)

email_task = Task(
    description=f"""Take the following rough email and rewrite it into a professional and polished version.
    Expand abbreviations:
    '''{original_email}'''""",
    agent=email_assistant,
    expected_output="A professional written email with proper formatting and content.",
)

crew = Crew(
    agents=[email_assistant],
    tasks=[email_task],
    verbose=True
)

# result = crew.kickoff()
# print(result)

# In Jupyter, you can use 'await' directly at the cell level!
result = await crew.kickoff_async()
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9a0a7bc4-dc2c-488e-a23f-99d04f41813e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the following rough email and rewrite it into a professional and polished version.                  │
│      Expand abbreviations:                                                                                      │
│      '''                                                                                                        │
│  looping in John. TA and PRG updates are in the deck. ETA for PRG integration is Fri.                           │
│  Let's sync up tomorrow. BTW, ping me if any hurdles.                                                           │
│  '''                                                                                                            │
│  ID: a81af331-82dd-47a6-a1a3-3388cc6acf02                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Email Assistant Agent                                                                                   │
│                                                                                                                 │
│  Task: Take the following rough email and rewrite it into a professional and polished version.                  │
│      Expand abbreviations:                                                                                      │
│      '''                                                                                                        │
│  looping in John. TA and PRG updates are in the deck. ETA for PRG integration is Fri.                           │
│  Let's sync up tomorrow. BTW, ping me if any hurdles.                                                           │
│  '''                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool jargon_replacement_tool executed with result: Consider replacing 'PRG' with 'Project Gemini'
Consider replacing 'BTW' with 'by the way'
Consider replacing 'ETA' with 'estimated time of arrival'
Consider replacing 'TA' with 'technical architecture...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: jargon_replacement_tool                                                                                  │
│  Args: {'email': "looping in John. TA and PRG updates are in the deck. ETA for PRG integration is Fri. Let's    │
│  sync up tomorrow. BTW, ping me if any hurdles."}                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Email Assistant Agent                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Project Status Update: Technical Architecture and Project Gemini Integration                          │
│                                                                                                                 │
│  Hi [Recipient Name],                                                                                           │
│                                                                                                                 │
│  I have added John to this email thread for visibility.                                                         │
│                                                                                                                 │
│  The latest updates regarding Technical Architecture and Project Gemini have been uploaded to the presentation  │
│  deck. Please note that the estimated completion date for the Project Gemini integration is this Friday.        │
│                                                                                                                 │
│  Let us connect tomorrow to discuss these updates further. By the way, please feel free to reach out to me if   │
│  you encounter any hurdles or roadblocks in the meantime.                                                       │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the following rough email and rewrite it into a professional and polished version.                  │
│      Expand abbreviations:                                                                                      │
│      '''                                                                                                        │
│  looping in John. TA and PRG updates are in the deck. ETA for PRG integration is Fri.                           │
│  Let's sync up tomorrow. BTW, ping me if any hurdles.                                                           │
│  '''                                                                                                            │
│  Agent: Email Assistant Agent                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Subject: Project Status Update: Technical Architecture and Project Gemini Integration

Hi [Recipient Name],

I have added John to this email thread for visibility.

The latest updates regarding Technical Architecture and Project Gemini have been uploaded to the presentation deck. Please note that the estimated completion date for the Project Gemini integration is this Friday.

Let us connect tomorrow to discuss these updates further. By the way, please feel free to reach out to me if you encounter any hurdles or roadblocks in the meantime.

Best regards,

[Your Name]


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9a0a7bc4-dc2c-488e-a23f-99d04f41813e                                                                       │
│  Final Output: Subject: Project Status Update: Technical Architecture and Project Gemini Integration            │
│                                                                                                                 │
│  Hi [Recipient Name],                                                                                           │
│                                                                                                                 │
│  I have added John to this email thread for visibility.                                                         │
│                                                                                                                 │
│  The latest updates regarding Technical Architecture and Project Gemini have been uploaded to the presentation  │
│  deck. Please note that the estimated completion date for the Project Gemini integration is this Friday.        │
│                                                                                                                 │
│  Let us connect tomorrow to discuss these updates further. By the way, please feel free to reach out to me if   │
│  you encounter any hurdles or roadblocks in the meantime.                                                       │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯